# ঠেলাগাড়ি Python ল্যাব — মডিউল ৪

**Ostad · Data Engineering Bootcamp · ব্যাচ ০১**

ক্লাস ৭ আর ৮-এর সব বিষয় এক জায়গায় — নিজের হাতে চালিয়ে শেখার জন্য।

| পর্ব | কী আছে | কোন ক্লাস |
|---|---|---|
| ১ | dictionary — গড়ন, পড়া, বদলানো, loop | ক্লাস ৭ |
| ২ | গোনা আর জমা করার প্যাটার্ন | ক্লাস ৭ |
| ৩ | record = dictionary, আর পুরো pipeline | ক্লাস ৭ |
| ৪ | CSV আর JSON — আসল ফাইল পড়া আর লেখা | ক্লাস ৮ |

> **আপডেট ২৩ নভেম্বর** — ক্লাস ৮-এর অংশটা (পর্ব ৪) যোগ করা হয়েছে।

---

### কীভাবে ব্যবহার করবেন

1. প্রতিটা কোডের ঘরের বাঁ পাশে ▶ বোতাম আছে। **ক্লিক করুন**, নয়তো **Shift + Enter**।
2. উপর থেকে নিচে **ক্রমে** চালাবেন।
3. 🖊 **নিজে চেষ্টা করুন** ঘরগুলোই আসল। ওখানে কোড লিখে চালাবেন।
4. আটকে গেলে নিচের **✅ উত্তর** ঘরটা খুলুন — কিন্তু অন্তত পাঁচ মিনিট নিজে চেষ্টার পরে।
5. ভেঙে গেলে **Runtime → Restart session**, তারপর আবার উপর থেকে।

> **মনে রাখবেন:** কোড পড়ে যাবেন না, **বদলে** দেখবেন। নাম পাল্টান, সংখ্যা পাল্টান,
> ইচ্ছা করে ভুল করুন।

---
## প্রস্তুতি — দুইটা ফাইল বানিয়ে নিন

ক্লাসের ডেটা ফাইল, আর গত মডিউলের `sales_tools.py`. **একবার চালালেই হবে।**

In [ ]:
raw_sales = """  ord100045 ; 2026-10-07 ;  RAHIM  uddin ; Miniket Chal 5kg ; 2 ; ৳ 1,240.50 ; BKASH ; delivered
ORD100046;2026-10-07;nusrat akter;Soybean Oil 2L;  1  ;৳350.00;nagad;delivered
  ord100047 ;2026-10-08;  Tanvir Hossain  ; Dim 12pc ;3; ৳ 441.75 ; Cash on Delivery ;DELIVERED
ORD100048 ; 2026-10-08 ; SUMAIYA  KHAN ; Ilish 1kg ;1;৳ 1,520.00; bkash ; cancelled
ord100049;2026-10-09; rakib  mia ;Broiler Murgi 1kg; 4 ;৳ 812.00;Card;delivered
  ORD100050 ;2026-10-09;  mim   begum ; Alu 2kg ; 2 ; ৳ 138.50 ; BKASH ;delivered
ord100051 ; 2026-10-10 ; FAHIM ahmed ; Chaa Pata 400g ; 1 ; ৳ 232.25 ; Rocket ; returned
ORD100052;2026-10-10;jannat  islam;Doodh 1L;  6  ; ৳ 558.00 ;nagad;delivered
  ord100053 ; 2026-10-11 ;  Sabbir   Rahman ; Peyaj 1kg ; 3 ;৳ 219.75; CASH ON DELIVERY ; delivered
ORD100054 ; 2026-10-11 ; afsana  chowdhury ; Apel 1kg ; 2 ; ৳ 1,105.00 ; bKash ; delivered"""

with open("raw_sales.txt", "w", encoding="utf-8") as fh:
    fh.write(raw_sales)

print("✅ raw_sales.txt তৈরি হয়েছে —", len(raw_sales.splitlines()), "লাইন")

আর গত মডিউলের `sales_tools.py`. এটা আপনারা নিজেরাই লিখেছিলেন — এখানে শুধু কপি করা।

In [ ]:
%%writefile sales_tools.py
"""ThelaGari-র record পরিষ্কার করার যন্ত্রপাতি (ক্লাস ৬-এর ফাইল)।"""


def clean_id(text):
    return text.strip().upper()


def clean_name(text):
    return " ".join(text.strip().title().split())


def clean_amount(text):
    text = text.replace("৳", "").replace(",", "").strip()
    return float(text)


def taka(amount):
    return f"৳{amount:.2f}"


def is_cancelled(status):
    return status.strip().lower() == "cancelled"

In [ ]:
import sales_tools

print(sales_tools.clean_id("  ord100045 "))
print(sales_tools.clean_name("mim   begum"))
print(sales_tools.taka(sales_tools.clean_amount(" ৳ 1,240.50 ")))

---
---
# পর্ব ১ — dictionary

## ১.১  কেন এই জিনিসটা দরকার

গত মডিউলের অ্যাসাইনমেন্টে payment ধরে হিসাব করতে এরকম লিখতে হয়েছিল:

```python
bkash_total  = 0.0
nagad_total  = 0.0
card_total   = 0.0
rocket_total = 0.0
cash_total   = 0.0
# ...আরো পাঁচটা count...

if payment == "bkash":
    bkash_total = bkash_total + amount
elif payment == "nagad":
    ...
# ...আরো তিনটা elif...
```

দশটা variable, পাঁচটা `elif`। আর কাল যদি **upay** নামে নতুন একটা payment
আসে? আবার কোড বদলাতে হবে।

আজকের পরে পুরোটা এই দুই লাইন:

```python
totals = {}
totals[payment] = totals.get(payment, 0) + amount
```

নতুন payment এলে **কোড বদলাতেই হবে না** — নতুন ঘর নিজে থেকে তৈরি হয়ে যাবে।

---
## ১.২  গড়ন

`list`-এ ঘর চেনা যায় **নম্বরে**। `dictionary`-তে **নামে**।

In [ ]:
# list — নম্বর দিয়ে
payments = ["bkash", "nagad", "card"]
print(payments[0])

# dictionary — নাম দিয়ে
order = {
    "order_id": "ORD100045",
    "amount": 1240.50,
    "payment": "bkash",
}
print(order["amount"])
print(order["payment"])

গড়নের চারটা অংশ:

| অংশ | কী |
|---|---|
| `{ }` | দ্বিতীয় বন্ধনী — list-এর `[ ]` না |
| `"order_id"` | **key** — ঘরের নাম, সাধারণত লেখা |
| `"ORD100045"` | **value** — ভেতরে যা রাখলেন, যেকোনো কিছু |
| `:` | কোলন দিয়ে নাম আর জিনিস আলাদা |

> **উপমা:** `list` হলো সারি ধরে দাঁড়ানো লোক — "তিন নম্বরে যে আছে" বলে ডাকতে হয়।
> `dictionary` হলো নামের তালিকা — সরাসরি নাম ধরে ডাকা যায়।

In [ ]:
print(len(order))        # কয়টা ঘর
print(type(order))

### 🖊 নিজে চেষ্টা করুন

একটা dictionary বানান `rider` নামে, তিনটা ঘর সহ — `name`, `phone`, `area`।
তারপর শুধু `area`-টা ছাপান।

In [ ]:
# এখানে লিখুন

In [ ]:
#@title ✅ উত্তর — ১.২  (চালাতে ▶ চাপুন)
rider = {
    "name": "Karim Mia",
    "phone": "01700000000",
    "area": "Dhanmondi",
}

print(rider["area"])

---
## ১.৩  পড়া — `[]` বনাম `.get()`

দুইটাই পড়ে। পার্থক্য দেখা যায় **ঘরটা না থাকলে**।

In [ ]:
print(order["amount"])          # আছে — ঠিকঠাক

try:
    print(order["discount"])    # নেই — ভেঙে যায়
except KeyError as e:
    print("KeyError:", e)
    print("→ 'discount' নামে কোনো ঘর নেই")

In [ ]:
print(order.get("discount"))        # None — ভাঙে না
print(order.get("discount", 0))     # না থাকলে যা দেবেন, তাই
print(order.get("amount", 0))       # থাকলে আসলটাই

| কখন | কোনটা |
|---|---|
| নিশ্চিত ঘরটা আছে | `order["amount"]` — না থাকলে ভেঙে পড়াই ভালো, ভুলটা তখনই ধরা পড়ে |
| ঘরটা না-ও থাকতে পারে | `order.get("x", 0)` — আজকের গোনার প্যাটার্ন এটার উপরেই দাঁড়ানো |

ঘরটা আছে কি না শুধু জানতে চাইলে:

In [ ]:
print("amount" in order)
print("discount" in order)

---
## ১.৪  যোগ, বদল, মোছা

In [ ]:
order["rider"] = "Karim"        # ছিল না — নতুন ঘর হয়ে গেল
print(order)

order["amount"] = 1300.00       # ছিল — বদলে গেল
print(order["amount"])

> **একই লাইনে দুইটা কাজ।** Python দেখে নেয় — ঘরটা আছে? তাহলে বদলে দাও।
> নেই? তাহলে বানিয়ে ফেলো।
>
> এই এক বাক্যটাই পরের অংশের ভিত।

In [ ]:
del order["rider"]              # মুছে ফেলা
print(order)

order.pop("rider", None)        # না থাকলেও ভাঙবে না
print("ভাঙেনি ✓")

---
## ১.৫  loop — তিনভাবে

In [ ]:
order = {"order_id": "ORD100045", "amount": 1240.50, "payment": "bkash"}

print("শুধু নাম:")
for k in order:
    print("   ", k)

print("\nশুধু জিনিস:")
for v in order.values():
    print("   ", v)

print("\nদুইটাই — সবচেয়ে বেশি লাগে:")
for k, v in order.items():
    print(f"    {k:<12} {v}")

> `k, v` — দুইটা নাম একসাথে। এই গড়নটা ক্লাস ৫-এ `enumerate()`-এ দেখেছিলেন।

সাজানো ক্রমে চাইলে `sorted()`:

In [ ]:
for k in sorted(order):
    print(f"{k:<12} {order[k]}")

---
---
# পর্ব ২ — গোনা আর জমা করা ⭐

আজকের আসল জিনিস। **এই দুই লাইন আপনি হাজারবার লিখবেন।**

## ২.১  আগে ভুলটা দেখুন

কেন `.get()` লাগে, সেটা ভুলটা দেখলেই বোঝা যায়।

In [ ]:
totals = {}

try:
    totals["bkash"] = totals["bkash"] + 1240.50
except KeyError as e:
    print("KeyError:", e)
    print("→ প্রথমবার ঘরটা তো নেই! যোগ করবো কীসের সাথে?")

## ২.২  এবার ঠিকটা

In [ ]:
totals = {}

totals["bkash"] = totals.get("bkash", 0) + 1240.50
print(totals)

totals["bkash"] = totals.get("bkash", 0) + 138.50
print(totals)

totals["nagad"] = totals.get("nagad", 0) + 350.00
print(totals)

### লাইনটা বাংলায় পড়ুন

> **"bkash নামের ঘরে যা আছে তা নাও — না থাকলে শূন্য ধরো — তার সাথে এই অর্ডারের
> টাকা যোগ করে আবার bkash নামের ঘরে রাখো।"**

ওই **"না থাকলে শূন্য ধরো"** অংশটাই প্রথমবারের সমস্যা মিটিয়ে দেয়। এটাই পুরো কৌশল।

### 🖊 নিজে চেষ্টা করুন

নিচের list-এ কোন ফল কতবার আছে, গুনুন। উত্তর আসার কথা:
`{'আম': 3, 'কলা': 2, 'পেয়ারা': 1}`

In [ ]:
fruits = ["আম", "কলা", "আম", "পেয়ারা", "আম", "কলা"]

counts = {}

# এখানে লিখুন

print(counts)

In [ ]:
#@title ✅ উত্তর — ২.২  (চালাতে ▶ চাপুন)
fruits = ["আম", "কলা", "আম", "পেয়ারা", "আম", "কলা"]

counts = {}
for f in fruits:
    counts[f] = counts.get(f, 0) + 1

print(counts)

---
---
# পর্ব ৩ — record = dictionary

## ৩.১  `parts[5]` আর মনে রাখতে হবে না

এতদিন আমরা লিখেছি `parts[5]`, `parts[6]`.

**প্রশ্ন:** `parts[6]`-এ কী ছিল, মনে আছে?

...মনে নেই, তাই না? এটাই সমস্যা। সংখ্যা দিয়ে ঘর চেনা মানে প্রতিবার ফাইল খুলে
গুনে দেখা।

In [ ]:
import sales_tools


def parse_record(raw):
    """একটা কাঁচা লাইন → পরিষ্কার dictionary"""
    parts = raw.split(";")
    return {
        "order_id":      sales_tools.clean_id(parts[0]),
        "order_date":    parts[1].strip(),
        "customer_name": sales_tools.clean_name(parts[2]),
        "item_name":     parts[3].strip(),
        "quantity":      int(parts[4].strip()),
        "amount":        sales_tools.clean_amount(parts[5]),
        "payment":       parts[6].strip().lower(),
        "status":        parts[7].strip().lower(),
    }


line = "  ord100045 ; 2026-10-07 ;  RAHIM  uddin ; Miniket Chal 5kg ; 2 ; ৳ 1,240.50 ; BKASH ; delivered"
rec = parse_record(line)

for k, v in rec.items():
    print(f"{k:<15} {v}")

এখন পড়লেই বোঝা যায়:

In [ ]:
print(rec["amount"])
print(rec["payment"])
print(rec["customer_name"])

> **সবচেয়ে বড় লাভটা:** `parts[5]` এখন **এক জায়গায়** লেখা — `parse_record`-এর
> ভেতরে। কাল ফাইলের গড়ন বদলালে আপনি শুধু ওই function-টা বদলাবেন, বাকি পুরো
> প্রোগ্রাম ছোঁবেন না।

---
## ৩.২  list-এর ভেতরে dictionary

দশটা record মানে দশটা dictionary-র একটা list।

In [ ]:
with open("raw_sales.txt", encoding="utf-8") as fh:
    lines = fh.read().strip().split("\n")

records = []
cancelled = 0

for raw in lines:
    r = parse_record(raw)
    if r["status"] == "cancelled":
        cancelled = cancelled + 1
        continue
    records.append(r)

print(f"মোট লাইন: {len(lines)}  ·  বাতিল: {cancelled}  ·  হিসাবে: {len(records)}")
print()
print("শূন্য নম্বর record-এর amount:", records[0]["amount"])
print("দুই নম্বর record-এর নাম:    ", records[2]["customer_name"])

> এই গড়নটা আপনি সারাজীবন দেখবেন। API থেকে যা আসে, JSON ফাইলে যা থাকে,
> database থেকে যা পড়বেন — প্রায় সবই **dictionary-র একটা list**।
>
> সোমবারের ক্লাসে JSON ধরবো, তখন দেখবেন ঠিক এই জিনিসটাই হাতে আসে।

---
## 🧪 মিনি-ল্যাব — একটা function, যেকোনো রিপোর্ট

এবার আজকের পুরস্কার। একটা function লিখবো যেটা **যেকোনো ঘর ধরে** হিসাব করতে পারে।

In [ ]:
def totals_by(records, key):
    """key যেটা দেবেন, সেই ঘর ধরে টাকা আর সংখ্যা জমা করে।"""
    totals = {}
    counts = {}

    for rec in records:
        name = rec[key]
        totals[name] = totals.get(name, 0.0) + rec["amount"]
        counts[name] = counts.get(name, 0) + 1

    return totals, counts


# payment ধরে
totals, counts = totals_by(records, "payment")

print("payment ধরে হিসাব")
for name in sorted(totals):
    print(f"   {name:<18} {counts[name]}টা   {sales_tools.taka(totals[name])}")
print(f"   {'সব মিলিয়ে':<18} {len(records)}টা   {sales_tools.taka(sum(totals.values()))}")

### এবার জাদুটা দেখুন

একটা শব্দ বদলান — `"payment"` → `"order_date"`. আর কিছু না।

In [ ]:
by_date, date_counts = totals_by(records, "order_date")

print("তারিখ ধরে হিসাব")
for day in sorted(by_date):
    print(f"   {day}   {date_counts[day]}টা   {sales_tools.taka(by_date[day])}")

একই function, পুরো অন্য একটা রিপোর্ট। **এটাই dictionary-র আসল শক্তি।**

গত সপ্তাহে এই দুইটা রিপোর্টের জন্য আপনাকে **পনেরোটা variable** লিখতে হতো।

### ✔ মিলিয়ে নিন

In [ ]:
assert len(records) == 9,                           "৯টা record হওয়ার কথা"
assert cancelled == 1,                              "বাতিল ১টা"
assert round(totals["bkash"], 2) == 2484.00,        "bkash ৳2484.00"
assert round(totals["nagad"], 2) == 908.00,         "nagad ৳908.00"
assert round(totals["card"], 2) == 812.00,          "card ৳812.00"
assert round(totals["rocket"], 2) == 232.25,        "rocket ৳232.25"
assert round(totals["cash on delivery"], 2) == 661.50, "cash ৳661.50"
assert round(sum(totals.values()), 2) == 5097.75,   "সব মিলিয়ে ৳5097.75"
assert round(by_date["2026-10-07"], 2) == 1590.50,  "০৭ তারিখে ৳1590.50"

print("🎉 সব মিলেছে!")
print()
print("৳5097.75 — এই সংখ্যাটা ক্লাস ৫, ৬ আর ৭-এ একই।")
print("কোড তিনবার বদলেছে, উত্তর বদলায়নি। এটাই ঠিক পথে থাকার প্রমাণ।")

---
## 🏁 চ্যালেঞ্জ ১ — সবচেয়ে বেশি টাকার দিন

`by_date` ঘুরে বের করুন কোন তারিখে সবচেয়ে বেশি টাকার অর্ডার হয়েছে।

💡 **ইঙ্গিত:** ক্লাস ৫-এর "সবচেয়ে বড়টা খোঁজা" নিয়মেই — একটা ঘর বাইরে রেখে
loop-এ মিলিয়ে দেখা। শুধু এবার list-এর বদলে dictionary ঘুরবেন।

**উত্তর আসার কথা:** `2026-10-07 — ৳1590.50`

In [ ]:
best_day = ""
best_money = 0.0

# এখানে লিখুন

print(best_day, sales_tools.taka(best_money))

In [ ]:
#@title ✅ উত্তর — চ্যালেঞ্জ ১  (চালাতে ▶ চাপুন)
best_day = ""
best_money = 0.0

for day in by_date:
    if by_date[day] > best_money:
        best_money = by_date[day]
        best_day = day

print(best_day, sales_tools.taka(best_money))

---
## 🏁 চ্যালেঞ্জ ২ — item ধরে হিসাব

`totals_by` ব্যবহার করে বের করুন কোন জিনিস কতবার বিক্রি হয়েছে, আর কত টাকার।

এটা এক লাইনের কাজ — function-টা তো লেখাই আছে।

In [ ]:
# এখানে লিখুন

In [ ]:
#@title ✅ উত্তর — চ্যালেঞ্জ ২  (চালাতে ▶ চাপুন)
by_item, item_counts = totals_by(records, "item_name")

for item in sorted(by_item):
    print(f"{item:<22} {item_counts[item]}টা   {sales_tools.taka(by_item[item])}")

> খেয়াল করলেন? নতুন কোনো হিসাবের দরকার হলে আপনি শুধু **key-র নামটা বদলান**।
> `"payment"`, `"order_date"`, `"item_name"`, `"status"` — যেকোনোটা।

---
---
# পর্ব ৪ — CSV আর JSON

**ক্লাস ৮**

এতক্ষণ যে ফাইলটা পড়েছি, সেটা আমি নিজের হাতে বানিয়েছিলাম — semicolon দিয়ে
আলাদা করা। শেখার জন্য ঠিক ছিল।

বাস্তবে data আসে দুইটা চেহারায় — **CSV** (টেবিলের মতো) আর **JSON**
(বাসার মতো)। এই পর্বে দুইটাই।

> উপরের পর্বগুলো আগে চালিয়ে নিন। এখানে `records`, `totals_by` আর
> `sales_tools` — তিনটাই লাগবে।

---
## ৪.১ — CSV জিনিসটা আসলে কী

কমা দিয়ে আলাদা করা কিছু লেখা। প্রথম সারিতে ঘরের নাম, তারপর প্রতি সারিতে
একটা record. ব্যস, এটুকুই।

নিচের ঘরটা চালিয়ে দেখুন একটা CSV ফাইল ভেতর থেকে কেমন দেখায়।

In [ ]:
sample = '''order_id,order_date,customer_name,amount,payment
ORD100045,2026-10-07,Rahim Uddin,1240.5,bkash
ORD100046,2026-10-07,Nusrat Akter,350.0,nagad'''

print(sample)
print()
print("সারি:", len(sample.split("\n")), "টা  (হেডার ধরে)")

খেয়াল করুন — **কোথাও লেখা নেই কোনটা সংখ্যা আর কোনটা লেখা।**

ফাইলের কাছে `1240.5`-ও অক্ষর, `Rahim Uddin`-ও অক্ষর। এই একটা কথা থেকেই
আজকের অর্ধেক ভুল জন্মায়। একটু পরে নিজের চোখে দেখবেন।

---
## ৪.২ — CSV লেখা

`csv` module Python-এর সাথেই আসে, আলাদা করে কিছু বসাতে হয় না।

লেখার দুইটা উপায় —

| কে | কী নেয় | কখন |
|---|---|---|
| `csv.writer` | list | ঘরের ক্রম আপনি মনে রাখবেন |
| `csv.DictWriter` | dictionary | নাম দেখে নিজেই ঠিক ঘরে বসায় |

আমাদের record তো আগে থেকেই dictionary — তাই **DictWriter**।

In [ ]:
import csv
import os

os.makedirs("output", exist_ok=True)

COLUMNS = ["order_id", "order_date", "customer_name", "item_name",
           "quantity", "amount", "payment", "status"]

with open("output/sales_clean.csv", "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=COLUMNS)
    writer.writeheader()
    for rec in records:
        writer.writerow(rec)

print("লেখা হলো — output/sales_clean.csv")
print()
print(open("output/sales_clean.csv", encoding="utf-8").read())

### তিনটা জিনিস খেয়াল করুন

1. **`newline=""`** — না দিলে Windows-এ প্রতি সারির পরে একটা ফাঁকা সারি পড়ে।
   Colab-এ টের পাবেন না, নিজের ল্যাপটপে পাবেন। তাই অভ্যাস করে ফেলুন।
2. **`encoding="utf-8"`** — বাংলা থাকলে এটা ছাড়া ভেঙে যাবে।
3. **`writeheader()`** — এটা না দিলে ঘরের নামগুলো ফাইলে যাবে না, আর পরে
   কেউ বুঝবে না কোন কলাম কী।

---
## ৪.৩ — CSV পড়া, আর আজকের ফাঁদ 🪤

`csv.DictReader` প্রতিটা সারিকে dictionary বানিয়ে দেয়। কোডটা দেখতে খুব
পরিষ্কার লাগে।

নিচের ঘরটা চালান — দেখতে সব ঠিকই লাগবে।

In [ ]:
with open("output/sales_clean.csv", encoding="utf-8", newline="") as fh:
    rows = list(csv.DictReader(fh))

print("সারি পেলাম:", len(rows), "টা")
print()
print(rows[0]["customer_name"], rows[0]["amount"])

দেখতে ঠিকই লাগছে, তাই না? এবার একটা লাইন যোগ করে দেখুন।

`repr()` মানে — **সাজিয়ে না, জিনিসটা আসলে কী সেটা দেখাও।**

In [ ]:
print(repr(rows[0]["amount"]))        # কোটেশন দুইটা খেয়াল করুন
print(type(rows[0]["amount"]))

### 🪤 ধরা পড়ল

`'1240.5'` — কোটেশনের ভেতরে। মানে এটা সংখ্যা না, **লেখা**।

এখন যোগ করতে গেলে কী হবে? নিচের ঘরটা চালিয়ে নিজের চোখে দেখুন।

In [ ]:
print("1240.5" + "350.0")      # যোগ হলো না, জোড়া লেগে গেল

try:
    print("1240.5" + 350.0)
except TypeError as e:
    print("TypeError:", e)

print(float("1240.5") + 350.0)  # এবার ঠিক

### সবচেয়ে ভয়ংকর কোনটা?

**প্রথমটা।** দ্বিতীয়টা তো চিৎকার করে বলে দিচ্ছে ভুল হয়েছে — আপনি তখনই ঠিক
করে ফেলবেন।

প্রথমটা চুপচাপ ভুল উত্তর দেয়। প্রোগ্রাম চলে, রিপোর্ট বেরোয়, আর সংখ্যাটা ভুল।

### সমাধান — পড়ার সাথে সাথেই বদলে নিন

In [ ]:
def read_clean_csv(path):
    records = []
    with open(path, encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            row["quantity"] = int(row["quantity"])      # পূর্ণসংখ্যা
            row["amount"] = float(row["amount"])        # দশমিকসহ সংখ্যা
            records.append(row)
    return records


back = read_clean_csv("output/sales_clean.csv")

print(repr(back[0]["amount"]), type(back[0]["amount"]))

### 🖊 নিজে চেষ্টা করুন

`back` থেকে payment ধরে হিসাব করুন — `totals_by` তো লেখাই আছে।

**৳5097.75 আসতেই হবে।** ফাইল ঘুরে এলেও সংখ্যা বদলায় না — সেটাই দেখার।

In [ ]:
# এখানে লিখুন

In [ ]:
#@title ✅ উত্তর — CSV থেকে হিসাব  (চালাতে ▶ চাপুন)
totals, counts = totals_by(back, "payment")

for name in sorted(totals):
    print(f"{name:<18} {counts[name]}টা   {sales_tools.taka(totals[name])}")

print(f"{'সব মিলিয়ে':<18} {len(back)}টা   {sales_tools.taka(sum(totals.values()))}")

---
## ৪.৪ — একটা কমা, আর পুরো ফাইল ভেঙে গেল

এখন বুঝবেন `csv` module কেন লাগে। নিজে `split(",")` করলে হয় না কেন।

নিচের ফাইলটায় একটা পণ্যের **নামের ভেতরেই একটা কমা** আছে।

In [ ]:
%%writefile items_messy.csv
item_code,item_name,unit_price,note
ITM-01,Miniket Chal 5kg,620.25,পুরনো দাম
ITM-02,"Chal, Miniket 5kg (new pack)",640.00,নামে কমা আছে
ITM-03,Soybean Oil 2L,350.00,

In [ ]:
line = open("items_messy.csv", encoding="utf-8").read().split("\n")[2]
print("লাইনটা:", line)
print()

# হাতে split
parts = line.split(",")
print("হাতে split করলে ঘর:", len(parts), "টা")
print("   নাম এসেছে:", parts[1])
print()

# csv module দিয়ে
with open("items_messy.csv", encoding="utf-8", newline="") as fh:
    rows = list(csv.DictReader(fh))
print("csv module-এ ঘর:", len(rows[1]), "টা")
print("   নাম এসেছে:", rows[1]["item_name"])

হেডারে ঘর **৪টা**। হাতে split করলে পেলাম **৫টা** — নামটা দুই টুকরা হয়ে
গেছে, আর দাম চলে গেছে ভুল ঘরে।

`csv` module কোটেশনের ভেতরের কমাটা চেনে, তাই নামটা আস্ত থাকে।

> নিজে split করলে একদিন না একদিন ভাঙবেই। প্রশ্ন শুধু কবে — আর ভাঙবে সেই
> দিন, যেদিন একজন গ্রাহকের নামে একটা কমা থাকবে।

---
## ৪.৫ — JSON

JSON দেখতে প্রায় Python-এর dictionary-র মতোই। আর সেটা কাকতালীয় না —
JSON পড়লে Python সেটাকে dictionary বানিয়েই দেয়।

CSV-র চেয়ে তিনটা জিনিস বেশি:

| | CSV | JSON |
|---|---|---|
| সংখ্যা সংখ্যাই থাকে? | না | **হ্যাঁ** |
| ভেতরে বাসা বানানো যায়? | না | **হ্যাঁ** |
| list রাখা যায়? | না | **হ্যাঁ** |

In [ ]:
import json

totals, counts = totals_by(back, "payment")
by_date, date_counts = totals_by(back, "order_date")

by_payment = {}
for name in sorted(totals):
    by_payment[name] = round(totals[name], 2)

by_day = {}
for day in sorted(by_date):
    by_day[day] = round(by_date[day], 2)

summary = {
    "company": "ThelaGari",
    "counted": len(back),
    "grand_total": round(sum(totals.values()), 2),
    "by_payment": by_payment,       # dictionary-র ভেতরে dictionary
    "by_date": by_day,
}

with open("output/summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)

print(open("output/summary.json", encoding="utf-8").read())

### লেখার সময় দুইটা জিনিস সবসময় দেবেন

- **`ensure_ascii=False`** — না দিলে বাংলা লেখা `\u098f` এরকম হয়ে যাবে
- **`indent=2`** — না দিলে পুরো ফাইলটা এক লাইনে, কেউ পড়তে পারবে না

নিচের ঘরটা চালিয়ে পার্থক্যটা নিজের চোখে দেখুন।

In [ ]:
small = {"নাম": "ঠেলাগাড়ি", "মোট": 5097.75}

print("ensure_ascii ছাড়া:")
print(json.dumps(small))
print()
print("ensure_ascii=False দিয়ে:")
print(json.dumps(small, ensure_ascii=False, indent=2))

> `dump` আর `dumps` — শেষের **s** মানে string. ফাইলে লিখতে `dump`,
> variable-এ রাখতে `dumps`. `load` আর `loads`-ও একই নিয়ম।

### JSON ফেরত পড়া — আর এখানেই আসল তফাত

CSV থেকে পড়ে `float()` করতে হয়েছিল। JSON-এ করতে হবে না।

In [ ]:
with open("output/summary.json", encoding="utf-8") as fh:
    loaded = json.load(fh)

print(loaded["grand_total"], type(loaded["grand_total"]))
print(loaded["by_payment"]["bkash"], type(loaded["by_payment"]["bkash"]))
print()
print("by_payment-এ ঘর:", len(loaded["by_payment"]), "টা")
print("CSV-তে এই বাসা বানানো যেত না।")

---
## ৪.৬ — কোনটা কখন

প্রশ্নটা একটাই: **data-টা কি টেবিলের মতো, না বাসার মতো?**

| নেবেন | যখন |
|---|---|
| **CSV** | সব সারিতে একই ঘর · কেউ Excel-এ খুলবে · লাখ লাখ সারি · database-এ ঢালবেন |
| **JSON** | ভেতরে গড়ন আছে · type ধরে রাখা জরুরি · API-তে যাবে বা আসবে · config বা সারাংশ |

বাস্তবে প্রায় সব pipeline-এ **দুইটাই** থাকে — বড় data CSV-তে, আর তার পাশে
একটা ছোট JSON: কবে চললো, কয়টা সারি পড়লো, কয়টা বাদ গেল।

---
## 🏁 চ্যালেঞ্জ — পুরো pipeline, শুরু থেকে শেষ

একটা function লিখুন যেটা:

1. `records` থেকে **item ধরে** হিসাব করবে
2. ফল একটা CSV-তে লিখবে — `output/item_summary.csv`, ঘর তিনটা:
   `item_name`, `orders`, `total_amount`
3. আর একটা JSON-এ লিখবে — `output/item_summary.json`
4. তারপর JSON-টা ফেরত পড়ে সবচেয়ে বেশি টাকার পণ্যটা ছাপাবে

এটাই Data Engineer-এর রোজকার কাজ, ছোট আকারে।

In [ ]:
# এখানে লিখুন

In [ ]:
#@title ✅ উত্তর — পুরো pipeline  (চালাতে ▶ চাপুন)
by_item, item_counts = totals_by(records, "item_name")

# ---- CSV ----
with open("output/item_summary.csv", "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=["item_name", "orders", "total_amount"])
    writer.writeheader()
    for name in sorted(by_item):
        writer.writerow({
            "item_name": name,
            "orders": item_counts[name],
            "total_amount": round(by_item[name], 2),
        })

# ---- JSON ----
report = {}
for name in sorted(by_item):
    report[name] = {
        "orders": item_counts[name],
        "total_amount": round(by_item[name], 2),
    }

with open("output/item_summary.json", "w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2)

# ---- ফেরত পড়া ----
with open("output/item_summary.json", encoding="utf-8") as fh:
    loaded = json.load(fh)

top_name = ""
top_money = 0.0
for name in loaded:
    if loaded[name]["total_amount"] > top_money:
        top_money = loaded[name]["total_amount"]
        top_name = name

print("দুইটা ফাইল লেখা হলো — output/ ফোল্ডারে")
print()
print(f"সবচেয়ে বেশি: {top_name} — {sales_tools.taka(top_money)}")

> **বাঁ পাশের 📁 ফোল্ডার আইকনে ক্লিক করুন** — `output/` খুলে ফাইলগুলো
> দেখতে পাবেন। ডাবল ক্লিক করলে Colab-এই খুলে যাবে।

---
---
## 📌 এক নজরে — যা শিখলেন

### dictionary (ক্লাস ৭)

| জিনিস | এক লাইনে |
|---|---|
| `{ }` | dictionary — নাম ধরে জিনিস রাখা |
| `d["k"]` | পড়া; ঘর না থাকলে **KeyError** |
| `d.get("k", 0)` | পড়া; ঘর না থাকলে যা দেবেন তাই |
| `d["k"] = v` | **আছে? বদলাও। নেই? বানাও।** একই লাইনে |
| `del d["k"]` · `.pop("k", None)` | মোছা |
| `"k" in d` | ঘরটা আছে কি না |
| `for k, v in d.items()` | নাম আর জিনিস একসাথে |
| `sorted(d)` | সাজানো ক্রমে |

### CSV আর JSON (ক্লাস ৮)

| জিনিস | এক লাইনে |
|---|---|
| `csv.DictWriter(fh, fieldnames=...)` | dictionary → CSV |
| `.writeheader()` · `.writerow(rec)` | হেডার, তারপর সারি |
| `csv.DictReader(fh)` | CSV → dictionary, **সব string** |
| `float(row["amount"])` | পড়ার সাথে সাথেই বদলে নিন |
| `newline=""` | না দিলে Windows-এ ফাঁকা সারি |
| `json.dump(d, fh, ensure_ascii=False, indent=2)` | dictionary → JSON |
| `json.load(fh)` | JSON → dictionary, **type বেঁচে থাকে** |
| `repr(x)` | জিনিসটা আসলে কী, সাজানো না |

---

### 🎯 পাঁচটা জিনিস, যেগুলো ভুললে চলবে না

1. **`d[k] = d.get(k, 0) + x`** — গোনার পুরো কাজটা এই এক লাইনে।
2. **`rec["amount"]` পড়লেই বোঝা যায়, `parts[5]` যায় না** — নাম দিন, নম্বর না।
3. **CSV-তে কোনো type নেই** — পড়ার সাথে সাথেই `float()`। না করলে error আসবে
   না, শুধু উত্তর ভুল হবে।
4. **নিজে `split(",")` করবেন না** — একটা কমা একদিন সব ভেঙে দেবে।
5. **JSON-এ গড়ন আর type দুইটাই বাঁচে** — তাই সারাংশ, config আর API-র data
   JSON-এ রাখি।

---

### 👉 এরপর কী

- **মডিউল ৪-এর অ্যাসাইনমেন্ট** — `module-04/assignments/module-04.md`,
  নতুন একটা data ফাইল নিয়ে, তিনটা লেভেল। জমা মডিউল ৫ শুরুর আগে।
- পরের ক্লাস **বুধবার ২৫ নভেম্বর** — মডিউল ৫ শুরু
- আটকে গেলে সাপোর্ট ক্লাস — রবি, মঙ্গল, বৃহস্পতি, রাত ৯টা

> একটা সংখ্যা খেয়াল করেছেন? **৳5097.75** — ক্লাস ৫, ৬, ৭ আর ৮, চারবারই একই।
> কোড বদলেছে, ফাইল বদলেছে, উত্তর বদলায়নি। ভালো কোডের সংজ্ঞা এটাই।